In [ ]:
# ── Config & project bootstrap ────────────────────────────────────────────────
import os, sys, glob, datetime
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.lines as mlines
import geopandas as gpd
import rioxarray  # noqa - registers .rio accessor on xarray DataArrays

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
PROJ = os.getcwd()
if os.path.join(PROJ, "src") not in sys.path:
    sys.path.insert(0, os.path.join(PROJ, "src"))

# ── Run to visualise ── test_results/<RUN_NAME>/best_mr*/predictions.pt
RUN_NAME = "v27"
EXCLUDE  = ["PFA"]

RESULTS_ROOT = os.path.join(PROJ, "test_results")
RESULTS_DIR  = os.path.join(RESULTS_ROOT, RUN_NAME)
REPORT_DIR   = os.path.join(PROJ, "report")

# swissBOUNDARIES3D shapefile (.shp.zip or unpacked .shp) - update for your machine
PATH_SWISSSHAPE = os.path.expanduser(
    os.environ.get("SWISSSHAPE",
        os.path.expanduser("~/Documents/ETH/_DAS Project/swissboundaries3d_2026-01_2056_5728.shp.zip"))
)

_CAND = [
    os.environ.get("DATA_ROOT", ""),   # set DATA_ROOT to override"/home/renku/work/PeakWeatherDataset",
         os.path.expanduser("~/Documents/ETH/_DAS Project/PeakWeatherDataset"),
         os.path.expanduser("~/PeakWeatherDataset"),
         os.path.join(PROJ, "PeakWeatherDataset")]
DATA_ROOT = next((p for p in _CAND if os.path.isdir(p)), _CAND[-1])

# ── Station x variable exclusions ───────────────────────────────────────────
# "all" drops the station entirely (redundant with EXCLUDE, asserted below);
# [list] drops only those variables at that station. Same convention as
# Test_Results_Exploration.ipynb's DROP_SV, so both notebooks agree.
DROP_SV = {
    "GES": ["pressure"],
    "PFA": "all",                       # already dropped by EXCLUDE, asserted below
    "LAE": ["wind_u", "wind_v"],        # "wind" = both components
}

# ── Variable metadata ───────────────────────────────────────────────────────
# (name, mae_column, physical unit, colormap, (vmin, vmax))
# Fixed (vmin, vmax) per variable -> every map in this notebook (different mask
# ratio, lead time, or model) reuses these bounds, so maps are comparable.
# plasma_r: yellow (low/good) -> magenta -> purple (high/bad)
VARIABLES = [
    ('temperature', 'temperature_mae', 'deg C', 'plasma_r', (0, 3.5)),
    ('pressure',    'pressure_mae',    'hPa',   'plasma_r', (0, 5.0)),
    ('humidity',    'humidity_mae',    '%',     'plasma_r', (0, 20.)),
    ('wind_u',      'wind_u_mae',      'm/s',   'plasma_r', (0, 3.0)),
    ('wind_v',      'wind_v_mae',      'm/s',   'plasma_r', (0, 3.0)),
]

# Bounds for absolute-delta maps (physical units, §13). Diverging around 0.
DELTA_BOUNDS = {
    'temperature': (-0.5, 2.5),
    'pressure':    (-0.8, 3.0),
    'humidity':    (-3.0, 12.0),
    'wind_u':      (-0.5, 2.0),
    'wind_v':      (-0.5, 2.0),
}

# Bootstrap defaults (§5, §14)
N_BOOT = 300
CI     = 0.90   # 90% confidence interval

print('PROJ        :', PROJ)
print('RUN_NAME    :', RUN_NAME)
print('RESULTS_DIR :', RESULTS_DIR)
print('DATA_ROOT   :', DATA_ROOT)
print('Contents    :', sorted(os.listdir(RESULTS_DIR)) if os.path.isdir(RESULTS_DIR) else '<missing>')

## Station list, physical σ, and DEM/topography

- `ds` — the one `load_peakweather()` / training use (10-min resolution). Gives the
  exact station ordering (`_resolve_keep_indices`) and per-station normalisation
  stats (`compute_obs_stats`) the model was trained with.
- `ds_topo` — a lightweight daily-resolution load with `extended_topo_vars='DEM'`,
  used only for the elevation raster and Swiss border. Must request the **same**
  `parameters` list as `load_peakweather()` — `PeakWeatherDataset` filters
  `stations_table` down to stations that have every requested parameter, so a
  shorter parameter list silently changes the station set.

In [ ]:
from data.dataset import load_peakweather, StationMAEDataset, compute_obs_stats

ds   = load_peakweather(root=DATA_ROOT)
keep = StationMAEDataset._resolve_keep_indices(ds, EXCLUDE)
STN  = [ds.stations_table.index[i] for i in keep]          # station ids, prediction order
N    = len(STN)

_st  = compute_obs_stats(ds, train_years=None, per_station=True)
STD  = np.clip(_st["std"].numpy()[keep][:, :5], 1e-6, None)   # (N, 5)
MEAN = _st["mean"].numpy()[keep][:, :5]                       # (N, 5)

_t = ds.stations_table.loc[STN]
coords = pd.DataFrame({
    "station_idx":    range(N),
    "nat_abbr":       STN,
    "e_lv95":         _t["swiss_easting"].astype(float).values,
    "n_lv95":         _t["swiss_northing"].astype(float).values,
    "station_height": _t["station_height"].astype(float).values,
})
print(f"Stations after excluding {EXCLUDE}: {N}")
coords.head()

In [ ]:
from peakweather.dataset import PeakWeatherDataset

# Same parameter list as load_peakweather() - see markdown above for why.
ds_topo = PeakWeatherDataset(
    root=DATA_ROOT,
    parameters=[
        "temperature",
        "pressure",
        "humidity",
        "wind_speed",
        "wind_direction",
        "precipitation",
    ],
    compute_uv=True,
    station_type="meteo_station",
    imputation_method=None,
    freq="d",
    extended_topo_vars="DEM",
)
assert len(ds_topo.stations_table) == len(ds.stations_table), (
    "ds_topo resolved a different station set than ds - DEM/topo station "
    "labels would not line up with the coords/STD computed above."
)
print(f"ds_topo stations: {len(ds_topo.stations_table)}  (DEM-ready)")

## Station × variable exclusions (`KEEP` mask)

`KEEP` is `(N, 5)` bool and gets ANDed into the validity mask inside
`per_station_mae_table()`, so every table/plot below inherits GES/pressure,
PFA/all, LAE/wind consistently.

In [ ]:
VAR_NAMES_5 = [v for v, *_ in VARIABLES]

KEEP, _rep = np.ones((N, len(VAR_NAMES_5)), bool), []
for _stn, _spec in DROP_SV.items():
    if _stn not in STN:
        if _stn in EXCLUDE:
            _rep.append(f"  {_stn:<4}  already excluded from the network by EXCLUDE")
            continue
        raise KeyError(f"DROP_SV names station {_stn!r}, which is neither in the "
                       f"modelled network nor in EXCLUDE={EXCLUDE}. Check the "
                       f"abbreviation against ds.stations_table.index.")
    _vs = list(VAR_NAMES_5) if _spec == "all" else list(_spec)
    _bad = [v for v in _vs if v not in VAR_NAMES_5]
    if _bad:
        raise KeyError(f"DROP_SV[{_stn!r}] names unknown variables {_bad}; "
                       f"VAR_NAMES_5={VAR_NAMES_5}")
    _si = STN.index(_stn)
    for _v in _vs:
        KEEP[_si, VAR_NAMES_5.index(_v)] = False
    _rep.append(f"  {_stn:<4}  station index {_si:3d}   dropped: {', '.join(_vs)}")

print("station x variable exclusions")
print("\n".join(_rep))
print(f"  cells kept: {KEEP.sum()}/{KEEP.size}"
      f"  ({KEEP.size - KEEP.sum()} station-variable pairs dropped)")

## Load predictions.pt and compute per-station metrics

`per_station_mae_table()` replaces the old `per_station_metrics.csv` read. Every
call below is filtered through `KEEP`; `delta_idx` and `masked_only` are the two
extra knobs used by the later sections. `_window_sums()` / `_bootstrap_ci_from_sums()`
are the building blocks §5 and §14 use for the bootstrap confidence intervals -
they precompute per-window sums so a bootstrap resample of windows is just an
array-index + sum, not a full recomputation.

In [ ]:
def per_station_mae_table(pred_dict, coords, keep_mask=None,
                          delta_idx=None, masked_only=False, window_sel=None,
                          visible_only=False):
    """
    Build a per-station MAE/RMSE table directly from a v27-style predictions dict
    (keys: preds, targets, masks, spatial, masked_idx, ...). Same accumulation as
    engine/evaluate.py::evaluate_per_station(), run against a saved dump instead
    of a live model + loader.

    Args:
        pred_dict:   dict loaded via torch.load(..., weights_only=False).
        coords:      DataFrame with station_idx, nat_abbr, e_lv95, n_lv95,
                     station_height.
        keep_mask:   (N, 5) bool - station x variable exclusions (KEEP). None
                     keeps everything.
        delta_idx:   None averages over ALL K lead times (evaluate_per_station's
                     own convention). An int fixes a single lead-time slot
                     (e.g. k=1 for +30min on the 13-lead v27 grid) so lead-time
                     maps are directly comparable to each other.
        visible_only: mr0.50 dumps only, the complement of masked_only —
                     restrict to (window, station) pairs where the station WAS
                     visible to the encoder. Pairing the two isolates the cost
                     of hiding a station from every other difference: same
                     model, same windows, same stations, same lead.
        masked_only: mr0.50 dumps only. If True, restrict to (window, station)
                     pairs where that station was actually hidden from the
                     encoder (pred_dict['masked_idx']) - isolates gap-filling
                     skill from the ~50% of windows where the station stayed
                     visible. Requires masked_idx with N_masked > 0.
        window_sel:  optional (M,) bool array selecting a subset of windows
                     (e.g. one season). None uses every window.

    Returns:
        DataFrame merged with coords, one row per station.
    """
    P  = pred_dict["preds"]                       # (M, K, N, 5) normalised
    T  = pred_dict["targets"][:, :, :, :5]        # (M, K, N, 5)
    Mk = pred_dict["masks"][:, :, :, :5] > 0.5    # (M, K, N, 5)

    if delta_idx is not None:
        P, T, Mk = P[:, delta_idx:delta_idx+1], T[:, delta_idx:delta_idx+1], Mk[:, delta_idx:delta_idx+1]

    if window_sel is not None:
        P, T, Mk = P[window_sel], T[window_sel], Mk[window_sel]

    err  = (P - T).abs().numpy()                  # (M', K', N, 5)
    sqe  = ((P - T) ** 2).numpy()
    mask = Mk.numpy()

    if keep_mask is not None:
        mask = mask & keep_mask[None, None, :, :]

    if masked_only and visible_only:
        raise ValueError("masked_only and visible_only are mutually exclusive")
    if masked_only or visible_only:
        midx = pred_dict.get("masked_idx")
        if midx is None or midx.shape[1] == 0:
            raise ValueError(f"{'masked_only' if masked_only else 'visible_only'}"
                             "=True but pred_dict has no masked_idx "
                             "(this dump has no mr0.50 masking).")
        if window_sel is not None:
            midx = midx[window_sel]
        N_ = mask.shape[2]
        sel = np.zeros((midx.shape[0], N_), bool)
        np.put_along_axis(sel, midx.numpy(), True, axis=1)
        if visible_only:
            sel = ~sel
        mask = mask & sel[:, None, :, None]        # broadcast over K', V

    n_valid = mask.sum(axis=(0, 1))                # (N, 5)
    sum_abs = (err * mask).sum(axis=(0, 1))        # (N, 5)
    sum_sq  = (sqe * mask).sum(axis=(0, 1))

    N_ = mask.shape[2]
    rows = []
    for n in range(N_):
        row = {"station_idx": n}
        for v, (var, col, unit, *_r) in enumerate(VARIABLES):
            nv = int(n_valid[n, v])
            row[f"{var}_n_samples"] = nv
            if nv == 0:
                row[col] = float("nan")
                row[f"{var}_rmse"] = float("nan")
                continue
            mae_norm  = sum_abs[n, v] / nv
            rmse_norm = (sum_sq[n, v] / nv) ** 0.5
            row[col]           = mae_norm  * STD[n, v]     # physical units
            row[f"{var}_rmse"] = rmse_norm * STD[n, v]

        m_n  = mask[:, :, n, :]
        n_ov = int(m_n.sum())
        if n_ov > 0:
            row["overall_mae_norm"]  = float((err[:, :, n, :] * m_n).sum() / n_ov)
            row["overall_rmse_norm"] = float(((sqe[:, :, n, :] * m_n).sum() / n_ov) ** 0.5)
        else:
            row["overall_mae_norm"]  = float("nan")
            row["overall_rmse_norm"] = float("nan")
        rows.append(row)

    df = pd.DataFrame(rows)
    df.insert(1, "nat_abbr", coords["nat_abbr"].values)
    df = df.merge(coords, on=["station_idx", "nat_abbr"], how="left")
    return df


def lead_index(pred_dict, minutes):
    """K-index of the lead-time slot equal to `minutes`, from delta_steps."""
    steps = pred_dict["delta_steps"][0].numpy()
    mins  = steps * 10
    hit   = np.where(mins == minutes)[0]
    if len(hit) == 0:
        raise ValueError(f"no lead-time slot at {minutes} min in this dump; "
                         f"available: {sorted(mins.tolist())}")
    return int(hit[0])


def _window_sums(pred_dict, keep_mask=None, delta_idx=None, masked_only=False, window_sel=None):
    """
    Precompute per-window, per-station, per-variable sum|err| and valid-count,
    collapsed over K (lead time). A bootstrap resample of *windows* then only
    needs to index+sum these arrays, preserving the within-window correlation
    across lead times instead of treating every (window, horizon) pair as an
    independent sample.

    Returns:
        SA: (M, N, 5) float - sum of |pred - target| per window/station/var
        NV: (M, N, 5) float - count of valid (window, horizon) entries
    """
    P  = pred_dict["preds"]
    T  = pred_dict["targets"][:, :, :, :5]
    Mk = pred_dict["masks"][:, :, :, :5] > 0.5
    if delta_idx is not None:
        P, T, Mk = P[:, delta_idx:delta_idx+1], T[:, delta_idx:delta_idx+1], Mk[:, delta_idx:delta_idx+1]
    if window_sel is not None:
        P, T, Mk = P[window_sel], T[window_sel], Mk[window_sel]

    err  = (P - T).abs().numpy()
    mask = Mk.numpy()
    if keep_mask is not None:
        mask = mask & keep_mask[None, None, :, :]

    if masked_only:
        midx = pred_dict.get("masked_idx")
        if midx is None or midx.shape[1] == 0:
            raise ValueError("masked_only=True but pred_dict has no masked_idx.")
        if window_sel is not None:
            midx = midx[window_sel]
        N_ = mask.shape[2]
        sel = np.zeros((midx.shape[0], N_), bool)
        np.put_along_axis(sel, midx.numpy(), True, axis=1)
        mask = mask & sel[:, None, :, None]

    SA = (err * mask).sum(axis=1)                       # (M, N, 5)
    NV = mask.sum(axis=1).astype(np.float64)             # (M, N, 5)
    return SA, NV


def _bootstrap_ci_from_sums(SA, NV, n_boot=N_BOOT, seed=0):
    """
    Resample the window axis (with replacement) n_boot times and recompute the
    per-station-variable MAE each time (normalised space - multiply by STD for
    physical units). Returns (n_boot, N, 5).
    """
    rng = np.random.default_rng(seed)
    M = SA.shape[0]
    out = np.empty((n_boot,) + SA.shape[1:], dtype=np.float64)
    for b in range(n_boot):
        idx = rng.integers(0, M, size=M)
        sa = SA[idx].sum(axis=0)
        nv = NV[idx].sum(axis=0)
        out[b] = np.divide(sa, nv, out=np.full_like(sa, np.nan), where=nv > 0)
    return out


mr_dirs = sorted(d for d in os.listdir(RESULTS_DIR) if d.startswith("best_mr"))
print(f"{RUN_NAME}: mask ratios available -> {mr_dirs}")

pred_mr0 = torch.load(os.path.join(RESULTS_DIR, "best_mr0.00", "predictions.pt"),
                      map_location="cpu", weights_only=False)
df_mr0 = per_station_mae_table(pred_mr0, coords, keep_mask=KEEP)
assert len(df_mr0) == N, f"station mismatch: table has {len(df_mr0)} rows, expected {N}"
assert pred_mr0["preds"].shape[2] == N, (
    f"predictions have {pred_mr0['preds'].shape[2]} stations but the dataset "
    f"resolved {N}. Check EXCLUDE={EXCLUDE}.")

pred_mr5, df_mr5, df_mr5_masked = None, None, None
if "best_mr0.50" in mr_dirs:
    pred_mr5 = torch.load(os.path.join(RESULTS_DIR, "best_mr0.50", "predictions.pt"),
                          map_location="cpu", weights_only=False)
    df_mr5        = per_station_mae_table(pred_mr5, coords, keep_mask=KEEP)
    df_mr5_masked = per_station_mae_table(pred_mr5, coords, keep_mask=KEEP, masked_only=True)
    print(f"mr0.00: {len(df_mr0)} stations   mr0.50: {len(df_mr5)} stations "
          f"(masked-only variant also computed)")
else:
    print(f"mr0.00: {len(df_mr0)} stations   (no mr0.50 dump for run '{RUN_NAME}')")

df_mr0[['station_idx','nat_abbr','e_lv95','n_lv95','station_height','temperature_mae']].head(6)

## Swiss map helpers

`scatter_metric()` takes either `vmin`/`vmax` (linear scale, the usual case) or a
`norm` override (used for the diverging, zero/one-centred scales in §13).

In [ ]:
def _load_dem_and_border(ds_topo, path_swissshape, coarsen=10):
    """Load DEM raster + CH border. Call once and reuse the result."""
    switzerland = gpd.read_file(
        path_swissshape,
        layer='swissBOUNDARIES3D_1_5_TLM_LANDESGEBIET',
    ).to_crs('EPSG:2056')
    minx, miny, maxx, maxy = switzerland.total_bounds

    topo   = ds_topo.load_topography()
    dem    = topo['topo_DEM'].dem
    dem_ch = dem.rio.clip(switzerland.geometry, switzerland.crs, drop=False)

    dem_bg = dem.coarsen(x=coarsen, y=coarsen, boundary='trim').mean()
    dem_fg = dem_ch.coarsen(x=coarsen, y=coarsen, boundary='trim').mean()

    dem_bg = dem_bg.sel(x=slice(minx, maxx), y=slice(miny, maxy))
    dem_fg = dem_fg.sel(x=slice(minx, maxx), y=slice(miny, maxy))
    return dem_bg, dem_fg, switzerland


def draw_dem(ax, dem_bg, dem_fg, switzerland):
    """Render DEM background + Swiss border onto ax."""
    norm = mcolors.Normalize(vmin=0, vmax=4500)
    dem_bg.plot(ax=ax, cmap='terrain', norm=norm, alpha=0.35,
                robust=True, add_labels=False, add_colorbar=False)
    dem_fg.plot(ax=ax, cmap='terrain', norm=norm,
                robust=True, add_labels=False, add_colorbar=False)
    switzerland.boundary.plot(ax=ax, color='white', linewidth=1.0)
    ax.axis('off')


def scatter_metric(ax, df, metric_col, cmap='plasma_r',
                   vmin=None, vmax=None, norm=None, size=60,
                   edgecolor='white', linewidth=0.4, zorder=5):
    """Scatter stations on ax, coloured by a continuous metric value.

    Pass either (vmin, vmax) for a linear scale, or `norm` (e.g.
    matplotlib.colors.TwoSlopeNorm) for a diverging scale - norm takes
    precedence if both are given.
    """
    valid = df.dropna(subset=['e_lv95', 'n_lv95', metric_col])
    kwargs = dict(cmap=cmap, s=size, edgecolors=edgecolor, linewidths=linewidth, zorder=zorder)
    if norm is not None:
        kwargs['norm'] = norm
    else:
        kwargs['vmin'] = vmin
        kwargs['vmax'] = vmax
    return ax.scatter(valid['e_lv95'], valid['n_lv95'], c=valid[metric_col], **kwargs)


def mark_significant(ax, df, sig_col, size=90):
    """Overlay a black ring on stations where `sig_col` is True."""
    sig = df[df[sig_col] == True].dropna(subset=['e_lv95', 'n_lv95'])
    if len(sig):
        ax.scatter(sig['e_lv95'], sig['n_lv95'], facecolors='none',
                   edgecolors='black', linewidths=1.4, s=size, zorder=9)
    return len(sig)


def draw_dem_grid(row_labels, col_specs, get_df, suptitle, save_path,
                  size=50, extra_per_cell=None, sharp_row_label=True):
    """
    Grid of DEM maps with ONE shared colour bar per column instead of one per
    panel - the scale is already fixed (VARIABLES / col_specs), so repeating a
    colour bar on every subplot is redundant clutter.

    Args:
        row_labels: list of row labels (left-hand annotation).
        col_specs:  list of (col_label, metric_col, unit, cmap, (vmin, vmax) or norm).
                    If the 5th element is a matplotlib Normalize instance, it is
                    used as `norm=`; otherwise treated as (vmin, vmax).
        get_df:     callable(row_label, col_label) -> DataFrame for that cell.
        extra_per_cell: optional callable(ax, row_label, col_label, df) called
                    after the scatter, e.g. to add a significance overlay.

    Returns (fig, axes).
    """
    n_rows, n_cols = len(row_labels), len(col_specs)
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(5.4 * n_cols, 4.6 * n_rows), squeeze=False)
    fig.suptitle(suptitle, fontsize=14, y=1.0 + 0.006 * n_rows)

    sc_by_col = [None] * n_cols
    for r, rlab in enumerate(row_labels):
        for c, spec in enumerate(col_specs):
            clab, col, unit, cmap, scale = spec
            ax = axes[r][c]
            draw_dem(ax, dem_bg, dem_fg, switzerland)
            df_cell = get_df(rlab, clab)
            if isinstance(scale, mcolors.Normalize):
                sc = scatter_metric(ax, df_cell, col, cmap=cmap, norm=scale, size=size)
            else:
                lo, hi = scale
                sc = scatter_metric(ax, df_cell, col, cmap=cmap, vmin=lo, vmax=hi, size=size)
            sc_by_col[c] = sc
            if extra_per_cell is not None:
                extra_per_cell(ax, rlab, clab, df_cell)
            if r == 0:
                ax.set_title(clab, fontsize=12)
            if c == 0 and sharp_row_label:
                ax.text(-0.06, 0.5, rlab, transform=ax.transAxes, rotation=90,
                       va='center', ha='center', fontsize=11, fontweight='bold')

    for c, spec in enumerate(col_specs):
        _, _, unit, _, _ = spec
        fig.colorbar(sc_by_col[c], ax=axes[:, c].tolist(), fraction=0.025, pad=0.01,
                    label=unit, shrink=0.85)

    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    return fig, axes


print('Loading DEM + border (first call takes ~30 s) ...')
dem_bg, dem_fg, switzerland = _load_dem_and_border(ds_topo, PATH_SWISSSHAPE)
print('Done.')

---
## 1 — Per-variable MAE maps: mr0.00 (all stations visible)

Averaged over all lead times. For a fixed-lead-time view see §12.

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(28, 5))
fig.suptitle(f'MAE by station - {RUN_NAME} mr0.00 (all stations visible, all lead times)', fontsize=14, y=1.01)

for ax, (var, col, unit, cmap, (lo, hi)) in zip(axes, VARIABLES):
    draw_dem(ax, dem_bg, dem_fg, switzerland)
    sc = scatter_metric(ax, df_mr0, col, cmap=cmap, vmin=lo, vmax=hi, size=55)
    plt.colorbar(sc, ax=ax, fraction=0.035, pad=0.02, label=unit)
    ax.set_title(var, fontsize=11)

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'map_mae_mr0.00.png'), dpi=150, bbox_inches='tight')
plt.show()

---
## 2 — Per-variable MAE maps: mr0.50, all windows (mixed visible + masked)

This mixes windows where a station was visible to the encoder and windows where it
was hidden (~50/50 on average per station) — it is the metric the model was
literally trained/evaluated under, but **not** the right number to isolate
gap-filling skill. See §3 for that.

In [ ]:
if df_mr5 is not None:
    fig, axes = plt.subplots(1, 5, figsize=(28, 5))
    fig.suptitle(f'MAE by station - {RUN_NAME} mr0.50, ALL windows (mixed visible/masked)', fontsize=14, y=1.01)

    for ax, (var, col, unit, cmap, (lo, hi)) in zip(axes, VARIABLES):
        draw_dem(ax, dem_bg, dem_fg, switzerland)
        sc = scatter_metric(ax, df_mr5, col, cmap=cmap, vmin=lo, vmax=hi, size=55)
        plt.colorbar(sc, ax=ax, fraction=0.035, pad=0.02, label=unit)
        ax.set_title(var, fontsize=11)

    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, 'map_mae_mr0.50_all.png'), dpi=150, bbox_inches='tight')
    plt.show()
else:
    print(f"No mr0.50 dump for run '{RUN_NAME}' - skipping.")

---
## 3 — Per-variable MAE maps: mr0.50, **masked stations only**

Restricted to (window, station) pairs where the station was actually hidden from
the encoder (`masked_idx`) — this is the model's true gap-filling / spatial
interpolation skill, not diluted by the windows where the station's own last
observation was available. Same fixed colour scale as §1/§2, so all three are
directly comparable.

In [ ]:
if df_mr5_masked is not None:
    fig, axes = plt.subplots(1, 5, figsize=(28, 5))
    fig.suptitle(f'MAE by station - {RUN_NAME} mr0.50, MASKED-ONLY (gap-filling skill)', fontsize=14, y=1.01)

    for ax, (var, col, unit, cmap, (lo, hi)) in zip(axes, VARIABLES):
        draw_dem(ax, dem_bg, dem_fg, switzerland)
        sc = scatter_metric(ax, df_mr5_masked, col, cmap=cmap, vmin=lo, vmax=hi, size=55)
        plt.colorbar(sc, ax=ax, fraction=0.035, pad=0.02, label=unit)
        ax.set_title(var, fontsize=11)

    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, 'map_mae_mr0.50_masked_only.png'), dpi=150, bbox_inches='tight')
    plt.show()

    _n_valid = pred_mr5['masked_idx'].shape[1]
    print(f"masked_idx: {_n_valid} stations hidden per window "
          f"({_n_valid/N*100:.0f}% of the {N}-station network)")
else:
    print(f"No mr0.50 dump (with masked_idx) for run '{RUN_NAME}' - skipping.")

---
## 4 — Overall RMSE map (mr0.00 vs mr0.50 all-windows)

In [ ]:
if df_mr5 is not None:
    fig, axes = plt.subplots(1, 2, figsize=(18, 6))
    for ax, (df, title) in zip(axes, [
        (df_mr0, f'Overall RMSE - {RUN_NAME} mr0.00 (no masking)'),
        (df_mr5, f'Overall RMSE - {RUN_NAME} mr0.50 (all windows, mixed)'),
    ]):
        draw_dem(ax, dem_bg, dem_fg, switzerland)
        sc = scatter_metric(ax, df, 'overall_rmse_norm', cmap='plasma_r', vmin=0.3, vmax=1.0, size=60)
        plt.colorbar(sc, ax=ax, fraction=0.035, pad=0.02, label='RMSE (norm.)')
        ax.set_title(title, fontsize=11)
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, 'map_overall_rmse.png'), dpi=150, bbox_inches='tight')
    plt.show()
else:
    fig, ax = plt.subplots(figsize=(9, 6))
    draw_dem(ax, dem_bg, dem_fg, switzerland)
    sc = scatter_metric(ax, df_mr0, 'overall_rmse_norm', cmap='plasma_r', vmin=0.3, vmax=1.0, size=60)
    plt.colorbar(sc, ax=ax, fraction=0.035, pad=0.02, label='RMSE (norm.)')
    ax.set_title(f'Overall RMSE - {RUN_NAME} mr0.00 (no mr0.50 dump for this run)', fontsize=11)
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, 'map_overall_rmse.png'), dpi=150, bbox_inches='tight')
    plt.show()

---
## 5 — Masking penalty map: Delta MAE = mr0.50 (masked-only) - mr0.00

**Red** = stations that are harder to predict when masked from the encoder.
**White / green** = stations the model reconstructs well even without seeing their
own input. Uses `df_mr5_masked` (not the mixed §2 table) — the mixed table still
includes the visible-window half for each station, which understates the true
cost of hiding it.

**Black ring = statistically significant** at the {CI:.0%} level: the window axis
is resampled with replacement `N_BOOT` times (paired between mr0.00 and mr0.50,
since both dumps come from the same test windows in the same order) and the delta
recomputed each time; a station is marked significant if its confidence interval
excludes zero. Unmarked stations' deltas are plausibly just noise from a limited
number of valid windows.

In [ ]:
if df_mr5_masked is not None:
    diff = df_mr0[['station_idx', 'e_lv95', 'n_lv95', 'station_height']].copy()
    for var, col, unit, _, _ in VARIABLES:
        diff[f'd_{col}'] = df_mr5_masked[col].values - df_mr0[col].values
    diff['d_overall_mae'] = df_mr5_masked['overall_mae_norm'].values - df_mr0['overall_mae_norm'].values

    # ── Bootstrap significance (paired resample of windows) ──────────────────
    SA0, NV0 = _window_sums(pred_mr0, keep_mask=KEEP)
    SA5, NV5 = _window_sums(pred_mr5, keep_mask=KEEP, masked_only=True)

    if SA0.shape[0] != SA5.shape[0]:
        print(f"mr0.00 ({SA0.shape[0]} windows) and mr0.50 ({SA5.shape[0]} windows) "
              f"have different counts - cannot pair-bootstrap; skipping significance overlay.")
        for var, col, unit, _, _ in VARIABLES:
            diff[f'sig_{col}'] = False
    else:
        rng = np.random.default_rng(0)
        M = SA0.shape[0]
        boot_delta = np.empty((N_BOOT, N, len(VAR_NAMES_5)))
        for b in range(N_BOOT):
            idx  = rng.integers(0, M, size=M)
            mae0 = np.divide(SA0[idx].sum(0), NV0[idx].sum(0),
                             out=np.full((N, len(VAR_NAMES_5)), np.nan), where=NV0[idx].sum(0) > 0)
            mae5 = np.divide(SA5[idx].sum(0), NV5[idx].sum(0),
                             out=np.full((N, len(VAR_NAMES_5)), np.nan), where=NV5[idx].sum(0) > 0)
            boot_delta[b] = (mae5 - mae0) * STD          # physical units
        lo_ci = np.nanpercentile(boot_delta, (1 - CI) / 2 * 100, axis=0)
        hi_ci = np.nanpercentile(boot_delta, (1 + CI) / 2 * 100, axis=0)
        sig   = (lo_ci > 0) | (hi_ci < 0)                # CI excludes zero
        for v, (var, col, unit, _, _) in enumerate(VARIABLES):
            diff[f'sig_{col}'] = sig[:, v]
        print(f"significant stations ({CI:.0%} CI excludes 0), out of {N}:")
        for v, (var, col, unit, _, _) in enumerate(VARIABLES):
            print(f"  {var:<12} {int(sig[:, v].sum()):3d}/{N}")

    fig, axes = plt.subplots(1, 5, figsize=(28, 5))
    fig.suptitle(f'{RUN_NAME} masking penalty DMAE = mr0.50(masked-only) - mr0.00  '
                f'(red = harder when masked; black ring = significant @ {CI:.0%})',
                fontsize=13, y=1.01)
    for ax, (var, col, unit, _, _) in zip(axes, VARIABLES):
        draw_dem(ax, dem_bg, dem_fg, switzerland)
        sc = scatter_metric(ax, diff, f'd_{col}', cmap='RdBu_r', vmin=-0.5, vmax=0.5, size=55)
        mark_significant(ax, diff, f'sig_{col}')
        plt.colorbar(sc, ax=ax, fraction=0.035, pad=0.02, label=unit)
        ax.set_title(var, fontsize=11)
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, 'map_masking_penalty.png'), dpi=150, bbox_inches='tight')
    plt.show()
else:
    print(f"No mr0.50 dump for run '{RUN_NAME}' - skipping.")
    diff = None

---
## 6 — Station altitude vs MAE scatter

Circles = mr0.00  |  Triangles = mr0.50 masked-only (if available).

In [ ]:
# ── Altitude vs MAE at three fixed lead times, masked vs visible ────────────
# Rows are lead times, columns variables. The y-axis is SHARED DOWN EACH
# COLUMN so the growth from +30min to +6h reads as a vertical shift; sharing
# across variables instead would be meaningless (deg C vs hPa vs %).
#
# Three series per panel, all from the SAME v27 weights:
#   mr0.00          every station visible — pure forecasting
#   mr0.50 visible  station visible, but half the network is hidden
#   mr0.50 masked   station hidden — the model must reconstruct it
# The visible/masked pair is drawn from ONE dump, so the gap between them is
# the cost of hiding a station and nothing else: same windows, same stations,
# same lead, same weights.
LEADS_SCATTER = [('30 min', 30), ('2 h', 120), ('6 h', 360)]
_SERIES = [
    #('mr0.00',         '#2E7D8C', 'o', 0.70),
    ('mr0.50 visible', '#4C8C6B', 's', 0.55),
    ('mr0.50 masked',  '#D9663D', '^', 0.55),
]

_scatter = {}
for _lab, _mins in LEADS_SCATTER:
    _k = lead_index(pred_mr0, _mins)
    e = {'mr0.00': per_station_mae_table(pred_mr0, coords, keep_mask=KEEP,
                                         delta_idx=_k)}
    if pred_mr5 is not None:
        _k5 = lead_index(pred_mr5, _mins)
        e['mr0.50 visible'] = per_station_mae_table(
            pred_mr5, coords, keep_mask=KEEP, delta_idx=_k5, visible_only=True)
        e['mr0.50 masked'] = per_station_mae_table(
            pred_mr5, coords, keep_mask=KEEP, delta_idx=_k5, masked_only=True)
    _scatter[_lab] = e
print("built per-lead tables:", {k: list(v) for k, v in _scatter.items()})

def _rho(a, b):
    """Spearman without scipy: Pearson on ranks."""
    m = np.isfinite(a) & np.isfinite(b)
    if m.sum() < 8:
        return np.nan
    ra = np.argsort(np.argsort(a[m])); rb = np.argsort(np.argsort(b[m]))
    return float(np.corrcoef(ra, rb)[0, 1])

_ALT_BINS = np.array([0, 500, 900, 1300, 1800, 2500, 4000])

nrow, ncol = len(LEADS_SCATTER), len(VARIABLES)
fig, axes = plt.subplots(nrow, ncol, figsize=(4.6 * ncol, 3.4 * nrow),
                         sharey='col', sharex=True)
_lim = {ci: 0.0 for ci in range(ncol)}
for ri, (lab, _) in enumerate(LEADS_SCATTER):
    for ci, (var, col, unit, _c1, _c2) in enumerate(VARIABLES):
        ax = axes[ri, ci]
        for name, colour, marker, alpha in _SERIES:
            df = _scatter[lab].get(name)
            if df is None:
                continue
            d = df.dropna(subset=['station_height', col])
            if not len(d):
                continue
            ax.scatter(d['station_height'], d[col], color=colour, alpha=alpha,
                       s=22, marker=marker, edgecolors='none', label=name)
            # binned median: the scatter alone hides the altitude trend
            bi = np.digitize(d['station_height'].values, _ALT_BINS[1:-1])
            xs, ys = [], []
            for b in range(len(_ALT_BINS) - 1):
                sel = bi == b
                if sel.sum() >= 4:
                    xs.append(np.median(d['station_height'].values[sel]))
                    ys.append(np.median(d[col].values[sel]))
            if len(xs) > 1:
                ax.plot(xs, ys, '-', color=colour, lw=2.0, alpha=.95, zorder=5)
            _lim[ci] = max(_lim[ci], np.nanpercentile(d[col].values, 99))
        # quantify the altitude relationship for the headline series
        _d = _scatter[lab].get('mr0.50 masked', _scatter[lab]['mr0.00'])
        _d = _d.dropna(subset=['station_height', col])
        ax.annotate(f"rho={_rho(_d['station_height'].values, _d[col].values):+.2f}",
                    xy=(0.97, 0.05), xycoords='axes fraction', ha='right',
                    fontsize=8, color='#444')
        ax.grid(alpha=0.3)
        if ri == 0:
            ax.set_title(f'{var}  [{unit}]', fontsize=11)
        if ri == nrow - 1:
            ax.set_xlabel('Altitude (m)', fontsize=9)
        if ci == 0:
            ax.set_ylabel(f'+{lab}\nMAE', fontsize=10)

for ci in range(ncol):
    hi = _lim[ci] * 1.08
    for ri in range(nrow):
        axes[ri, ci].set_ylim(0, hi if hi > 0 else None)
axes[0, 0].legend(fontsize=8, loc='upper left', framealpha=.9)
fig.suptitle(f'{RUN_NAME} — station altitude vs MAE at fixed lead times\n'
             'lines = binned medians · rho annotated for the masked series · '
             'y-scale shared per variable across leads', fontsize=12, y=1.005)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'scatter_altitude_mae_by_lead.png'),
            dpi=150, bbox_inches='tight')
plt.show()

# masking penalty vs altitude, same three leads
if pred_mr5 is not None:
    fig, axes = plt.subplots(1, ncol, figsize=(4.4 * ncol, 3.4), sharex=True)
    for ci, (var, col, unit, _c1, _c2) in enumerate(VARIABLES):
        ax = axes[ci]
        for (lab, _), style in zip(LEADS_SCATTER, ['-', '--', ':']):
            a = _scatter[lab]['mr0.50 masked'].set_index('station_idx')[col]
            b = _scatter[lab]['mr0.50 visible'].set_index('station_idx')[col]
            pen = (a - b)
            h = _scatter[lab]['mr0.50 masked'].set_index('station_idx')['station_height']
            d = pd.DataFrame({'h': h, 'p': pen}).dropna()
            bi = np.digitize(d['h'].values, _ALT_BINS[1:-1])
            xs, ys = [], []
            for bb in range(len(_ALT_BINS) - 1):
                sel = bi == bb
                if sel.sum() >= 4:
                    xs.append(np.median(d['h'].values[sel]))
                    ys.append(np.median(d['p'].values[sel]))
            ax.plot(xs, ys, style, marker='o', ms=3, lw=1.6, label=f'+{lab}')
        ax.axhline(0, color='k', lw=.9)
        ax.set_title(f'{var}  [{unit}]', fontsize=10)
        ax.set_xlabel('Altitude (m)', fontsize=9); ax.grid(alpha=.3)
    axes[0].set_ylabel('masking penalty\nMAE(masked) - MAE(visible)', fontsize=9)
    axes[-1].legend(fontsize=8)
    fig.suptitle(f'{RUN_NAME} — cost of hiding a station, by altitude and lead '
                 '(binned medians)', fontsize=12, y=1.04)
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, 'scatter_altitude_masking_penalty.png'),
                dpi=150, bbox_inches='tight')
    plt.show()

---
## 7 — Masked stations example (one window)

Shows which stations the encoder sees vs. which it must reconstruct for one test
window, using the dump's real `masked_idx` tensor.

In [ ]:
WINDOW_IDX = 42    # change to inspect other windows

if pred_mr5 is not None and pred_mr5.get("masked_idx") is not None and pred_mr5["masked_idx"].shape[1] > 0:
    y_mask = pred_mr5['masks']            # (M, K, N, V)
    N_ = y_mask.shape[2]

    has_data   = y_mask[WINDOW_IDX, 0, :, :5].any(dim=-1).numpy()
    masked_set = set(pred_mr5['masked_idx'][WINDOW_IDX].tolist())
    n_masked   = len(masked_set)

    e_all = df_mr5['e_lv95'].values
    n_all = df_mr5['n_lv95'].values

    status = []
    for i in range(len(df_mr5)):
        if not has_data[i]:    status.append('no_data')
        elif i in masked_set:  status.append('masked')
        else:                  status.append('visible')
    status = np.array(status)

    fig, ax = plt.subplots(figsize=(11, 8))
    draw_dem(ax, dem_bg, dem_fg, switzerland)

    for grp, color, marker, sz, lab, zo in [
        ('visible', '#2196F3', 'o', 60, 'Visible (context)',       8),
        ('masked',  '#F44336', 'X', 75, 'Masked (model predicts)', 10),
        ('no_data', '#AAAAAA', 'o', 30, 'No sensor data',          6),
    ]:
        sel = status == grp
        ax.scatter(e_all[sel], n_all[sel], color=color, marker=marker, s=sz,
                   edgecolors='white', linewidths=0.5, label=lab, zorder=zo)

    ax.legend(fontsize=9, framealpha=0.85, loc='upper left')
    ax.set_title(f'{RUN_NAME} encoder masking (real masked_idx) - window #{WINDOW_IDX}   '
                 f'({n_masked}/{N_} stations hidden)', fontsize=11)
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, 'map_masked_stations_example.png'), dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Visible: {(status=="visible").sum()}  '
          f'Masked: {(status=="masked").sum()}  '
          f'No data: {(status=="no_data").sum()}')
else:
    print(f"No mr0.50 dump (with masked_idx) for run '{RUN_NAME}' - skipping.")

---
## 8 — Best / worst stations by temperature MAE

In [ ]:
TOP_N = 10
base_cols = ['nat_abbr', 'station_idx', 'station_height',
             'temperature_mae', 'temperature_rmse', 'overall_mae_norm']

_dfs = [('mr0.00', df_mr0)]
if df_mr5_masked is not None:
    _dfs.append(('mr0.50 masked-only', df_mr5_masked))

for label, df in _dfs:
    sub = df.dropna(subset=['temperature_mae']).sort_values('temperature_mae')
    print(f'\n-- {label} Top-{TOP_N} BEST temperature MAE')
    display(sub[base_cols].head(TOP_N).reset_index(drop=True))
    print(f'\n-- {label} Top-{TOP_N} WORST temperature MAE')
    display(sub[base_cols].tail(TOP_N).reset_index(drop=True))

---
## 9 — Humidity RMSE spatial pattern

In [ ]:
_panels = [(df_mr0, f'Humidity RMSE - {RUN_NAME} mr0.00')]
if df_mr5_masked is not None:
    _panels.append((df_mr5_masked, f'Humidity RMSE - {RUN_NAME} mr0.50 masked-only'))

fig, axes = plt.subplots(1, len(_panels), figsize=(9 * len(_panels), 6), squeeze=False)
axes = axes[0]
for ax, (df, title) in zip(axes, _panels):
    draw_dem(ax, dem_bg, dem_fg, switzerland)
    sc = scatter_metric(ax, df, 'humidity_rmse', cmap='RdPu', vmin=0, vmax=30, size=60)
    plt.colorbar(sc, ax=ax, fraction=0.035, pad=0.02, label='%')
    ax.set_title(title, fontsize=11)
plt.suptitle('Humidity RMSE (%)', fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'map_humidity_rmse.png'), dpi=150, bbox_inches='tight')
plt.show()

---
## 10 — Raw prediction vs target: time series for best and worst station

In [ ]:
VAR_IDX   = 0   # 0=temperature  1=pressure  2=humidity  3=wind_u  4=wind_v
DELTA_IDX = 1   # K index: 0=0 min  1=30 min  2=60 min ...
VAR_NAME  = 'temperature'

_pred_ts, _df_ts, _tag_ts = ((pred_mr5, df_mr5_masked, 'mr0.50 masked-only')
                             if pred_mr5 is not None else (pred_mr0, df_mr0, 'mr0.00'))

preds_t   = _pred_ts['preds']    # (M, K, N, V)
targets_t = _pred_ts['targets']  # (M, K, N, V)
masks_t   = _pred_ts['masks']    # (M, K, N, V)

sub = _df_ts.dropna(subset=['temperature_mae'])
best_idx  = int(sub.nsmallest(1, 'temperature_mae')['station_idx'].values[0])
worst_idx = int(sub.nlargest(1,  'temperature_mae')['station_idx'].values[0])

for idx, tag in [(best_idx, 'Best'), (worst_idx, 'Worst')]:
    row = sub[sub['station_idx'] == idx].iloc[0]
    abbr = row.get('nat_abbr', f'idx={idx}')
    print(f'{tag}: {abbr}  |  {row["station_height"]:.0f} m  '
          f'|  temperature_mae={row["temperature_mae"]:.3f} deg C')

fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

for ax, (idx, title_tag) in zip(axes, [(best_idx, 'Best'), (worst_idx, 'Worst')]):
    row  = sub[sub['station_idx'] == idx].iloc[0]
    abbr = row.get('nat_abbr', f'idx={idx}')

    pred_vals   = preds_t[:, DELTA_IDX, idx, VAR_IDX].numpy()
    target_vals = targets_t[:, DELTA_IDX, idx, VAR_IDX].numpy()
    valid       = masks_t[:, DELTA_IDX, idx, VAR_IDX].numpy().astype(bool)

    x = np.arange(len(pred_vals))
    ax.plot(x[valid], target_vals[valid], color='#CCCCCC', lw=1.4, alpha=0.95, label='target')
    ax.plot(x[valid], pred_vals[valid],   color='#FF4444', lw=1.1, alpha=0.9,
            linestyle='--', label='pred')
    mae = np.abs(pred_vals[valid] - target_vals[valid]).mean()
    ax.set_title(f'{title_tag}: {abbr}  ({row["station_height"]:.0f} m)  '
                 f'temperature @30 min  MAE={mae:.3f} (norm.)', fontsize=10)
    ax.set_ylabel('Norm. value')
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)

axes[-1].set_xlabel('Test window index')
plt.suptitle(f'{VAR_NAME} predictions - {RUN_NAME} {_tag_ts} - {len(x)} windows', fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'timeseries_best_worst.png'), dpi=150, bbox_inches='tight')
plt.show()

---
## 11 — Persistence baseline comparison

Skill score = 1 - model_RMSE / persistence_RMSE. Positive (blue) = model beats
persistence.

In [ ]:
def _per_station_rmse_from_tensors(pred_dict, label='', keep_mask=None):
    """Per-station overall RMSE, lead times k=1..K-1 (excludes nowcast k=0)."""
    preds   = pred_dict['preds']
    targets = pred_dict['targets'][:, :, :, :5]
    masks   = pred_dict['masks'  ][:, :, :, :5]

    persist = targets[:, 0:1, :, :]

    model_sq   = (preds   [:, 1:, :, :] - targets[:, 1:, :, :]) ** 2
    persist_sq = (persist.expand_as(preds[:, 1:, :, :]) - targets[:, 1:, :, :]) ** 2
    mask_fwd   = masks[:, 1:, :, :].bool().numpy()
    if keep_mask is not None:
        mask_fwd = mask_fwd & keep_mask[None, None, :, :]
    mask_fwd = torch.from_numpy(mask_fwd)

    N_ = preds.shape[2]
    rows_m, rows_p = [], []
    for n in range(N_):
        rm = {'station_idx': n}
        rp = {'station_idx': n}
        sq_sum_m = sq_sum_p = 0.0
        n_vars = 0
        for v, var in enumerate(VAR_NAMES_5):
            m_nv = mask_fwd[:, :, n, v]
            cnt  = int(m_nv.sum().item())
            if cnt > 0:
                ms = float(model_sq  [:, :, n, v][m_nv].mean().item())
                ps = float(persist_sq[:, :, n, v][m_nv].mean().item())
                rm[f'{var}_rmse_norm'] = ms ** 0.5
                rp[f'{var}_rmse_norm'] = ps ** 0.5
                sq_sum_m += ms
                sq_sum_p += ps
                n_vars   += 1
            else:
                rm[f'{var}_rmse_norm'] = float('nan')
                rp[f'{var}_rmse_norm'] = float('nan')
        rm['overall_rmse_norm'] = (sq_sum_m / n_vars) ** 0.5 if n_vars else float('nan')
        rp['overall_rmse_norm'] = (sq_sum_p / n_vars) ** 0.5 if n_vars else float('nan')
        rows_m.append(rm)
        rows_p.append(rp)

    df_m = pd.DataFrame(rows_m).merge(coords, on='station_idx', how='left')
    df_p = pd.DataFrame(rows_p).merge(coords, on='station_idx', how='left')
    if label:
        print(f'{label} - overall RMSE (norm, k>0)  '
              f'model median={df_m["overall_rmse_norm"].median():.3f}  '
              f'persist median={df_p["overall_rmse_norm"].median():.3f}')
    return df_m, df_p


_runs11 = [('mr0.00', pred_mr0)]
if pred_mr5 is not None:
    _runs11.append(('mr0.50 (all windows)', pred_mr5))

_results11 = {}
for tag, pred in _runs11:
    df_m, df_p = _per_station_rmse_from_tensors(pred, tag, keep_mask=KEEP)
    skill_col = df_m[['station_idx', 'e_lv95', 'n_lv95', 'nat_abbr']].copy()
    skill_col['skill'] = 1.0 - df_m['overall_rmse_norm'].values / df_p['overall_rmse_norm'].values
    _results11[tag] = (df_m, df_p, skill_col)

_all_rmse = pd.concat(
    [df_m['overall_rmse_norm'] for df_m, _, _ in _results11.values()]
    + [df_p['overall_rmse_norm'] for _, df_p, _ in _results11.values()]
).dropna()
RMSE_VMIN = float(_all_rmse.quantile(0.02))
RMSE_VMAX = float(_all_rmse.quantile(0.98))
print(f'Shared RMSE color scale: [{RMSE_VMIN:.3f}, {RMSE_VMAX:.3f}]')

fig, axes = plt.subplots(len(_results11), 3, figsize=(27, 6 * len(_results11)), squeeze=False)
fig.suptitle(
    f'{RUN_NAME} per-station RMSE - model vs persistence (lead times 30 min - 6 h)\n'
    'Skill = 1 - model/persistence  (blue = model better  |  red = persistence better)',
    fontsize=13, y=1.01
)

for row_idx, (tag, (df_m, df_p, df_sk)) in enumerate(_results11.items()):
    ax = axes[row_idx, 0]
    draw_dem(ax, dem_bg, dem_fg, switzerland)
    sc = scatter_metric(ax, df_m, 'overall_rmse_norm', cmap='plasma_r',
                        vmin=RMSE_VMIN, vmax=RMSE_VMAX, size=60)
    plt.colorbar(sc, ax=ax, fraction=0.035, pad=0.02, label='RMSE (norm.)')
    ax.set_title(f'Model - {tag}', fontsize=11)

    ax = axes[row_idx, 1]
    draw_dem(ax, dem_bg, dem_fg, switzerland)
    sc = scatter_metric(ax, df_p, 'overall_rmse_norm', cmap='plasma_r',
                        vmin=RMSE_VMIN, vmax=RMSE_VMAX, size=60)
    plt.colorbar(sc, ax=ax, fraction=0.035, pad=0.02, label='RMSE (norm.)')
    ax.set_title(f'Persistence - {tag}', fontsize=11)

    ax = axes[row_idx, 2]
    draw_dem(ax, dem_bg, dem_fg, switzerland)
    sc = scatter_metric(ax, df_sk, 'skill', cmap='RdBu', vmin=-0.3, vmax=0.3, size=60)
    plt.colorbar(sc, ax=ax, fraction=0.035, pad=0.02, label='Skill score')
    ax.set_title(f'Skill score - {tag}', fontsize=11)

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'map_persistence_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()

print('\nMedian per-station skill score (1 - model/persistence):')
for tag, (_, _, df_sk) in _results11.items():
    print(f'  {tag:22s} {df_sk["skill"].median():+.3f}   '
          f'({(df_sk["skill"] > 0).sum()}/{len(df_sk)} stations beat persistence)')

---
## 12 — Fixed lead-time MAE maps: 30 min / 2 h / 6 h

Same station, same fixed colour scale (from `VARIABLES`) as every other map in
this notebook, three lead times — lets error growth with lead time be read off
directly, without it being averaged away by §1's all-horizon table. One shared
colour bar per variable/column via `draw_dem_grid()`.

In [ ]:
LEAD_LABELS = [('30 min', 30), ('2 h', 120), ('6 h', 360)]
_idx0 = {lab: lead_index(pred_mr0, mins) for lab, mins in LEAD_LABELS}
_row_labels12 = [lab for lab, _ in LEAD_LABELS]

df_by_lead_mr0 = {lab: per_station_mae_table(pred_mr0, coords, keep_mask=KEEP, delta_idx=k)
                  for lab, k in _idx0.items()}

_col_specs12 = [(var, col, unit, cmap, (lo, hi)) for var, col, unit, cmap, (lo, hi) in VARIABLES]

draw_dem_grid(
    row_labels=_row_labels12,
    col_specs=_col_specs12,
    get_df=lambda rlab, clab: df_by_lead_mr0[rlab],
    suptitle=f'{RUN_NAME} mr0.00 - MAE by station at fixed lead times (shared colour scale, one bar per column)',
    save_path=os.path.join(RESULTS_DIR, 'map_mae_by_leadtime.png'),
    size=50,
)

---
## 13 — Error growth with lead time, by season — v27

Two views, side by side sections: the **ratio** MAE(2h)/MAE(30min) (how many
times worse the 2h forecast is), and the **absolute delta** MAE(2h) - MAE(30min)
(how much worse in the variable's own units) — the ratio alone can be misleading
at very-low-MAE stations, where a tiny 30-min denominator inflates the ratio even
if the 2h error itself is unremarkable; the delta map fixes that.

Both use a **diverging** colour scale centred on "no change" (ratio = 1, delta =
0) instead of a sequential one, so improvement (ratio < 1 / delta < 0) is visually
distinct from degradation, not clipped to the same floor colour.

In [ ]:
def _season_of_hours(hours):
    ep = datetime.datetime(1970, 1, 1)
    m  = np.array([(ep + datetime.timedelta(hours=float(h))).month for h in hours])
    return np.select(
        [np.isin(m, [12, 1, 2]), np.isin(m, [3, 4, 5]), np.isin(m, [6, 7, 8])],
        ["DJF", "MAM", "JJA"], default="SON",
    )

SEASONS = ["DJF", "MAM", "JJA", "SON"]
_season_labels = _season_of_hours(pred_mr0["window_hours"].numpy())
_k30, _k2h = _idx0["30 min"], _idx0["2 h"]

RATIO_TABLES = {}
for season in SEASONS:
    sel = _season_labels == season
    df_30 = per_station_mae_table(pred_mr0, coords, keep_mask=KEEP, delta_idx=_k30, window_sel=sel)
    df_2h = per_station_mae_table(pred_mr0, coords, keep_mask=KEEP, delta_idx=_k2h, window_sel=sel)

    t = df_30[['station_idx', 'nat_abbr', 'e_lv95', 'n_lv95', 'station_height']].copy()
    t['n_windows'] = int(sel.sum())
    for var, col, unit, _, _ in VARIABLES:
        t[f'ratio_{var}'] = df_2h[col].values / df_30[col].values
        t[f'delta_{var}'] = df_2h[col].values - df_30[col].values
    RATIO_TABLES[season] = t

print("Median MAE(2h)/MAE(30min) ratio and MAE(2h)-MAE(30min) delta, by season:")
for season in SEASONS:
    r = RATIO_TABLES[season]
    ratio_str = ", ".join(f"{var}={r[f'ratio_{var}'].median():.2f}" for var, *_ in VARIABLES)
    delta_str = ", ".join(f"{var}={r[f'delta_{var}'].median():+.2f}" for var, *_ in VARIABLES)
    print(f"  {season} (n={int(r['n_windows'].iloc[0])}):")
    print(f"    ratio: {ratio_str}")
    print(f"    delta: {delta_str}")

### 13a — Ratio MAE(2h) / MAE(30min), by season

In [ ]:
_ratio_norm = mcolors.TwoSlopeNorm(vmin=0.3, vcenter=1.0, vmax=4.0)
_col_specs13a = [(var, f'ratio_{var}', 'ratio', 'RdBu_r', _ratio_norm) for var, *_ in VARIABLES]

draw_dem_grid(
    row_labels=SEASONS,
    col_specs=_col_specs13a,
    get_df=lambda rlab, clab: RATIO_TABLES[rlab],
    suptitle=f'{RUN_NAME} mr0.00 - MAE(2h)/MAE(30min) ratio by season  '
             '(blue = improves with lead time, red = degrades, white ~ 1 = no change)',
    save_path=os.path.join(RESULTS_DIR, 'map_error_growth_ratio_by_season.png'),
    size=50,
)

---
## 14 — Multi-model comparison at a fixed 2 h lead time (mr0.00)

Discovers every run under `test_results/*/best_mr0.00/predictions.pt` and plots
the same fixed-lead-time MAE map for each, all on the shared `VARIABLES` colour
scale (one bar per column), so models are directly comparable station-by-station.
§14b adds a **winner map**: for each station and variable, which run had the
lowest MAE, with a bootstrap check for whether the win is real or a coin flip.

In [ ]:
_run_paths = sorted(glob.glob(os.path.join(RESULTS_ROOT, "*", "best_mr0.00", "predictions.pt")))
_runs14 = [(p.split(os.sep)[-3], p) for p in _run_paths]
print("runs found for §14:", [r for r, _ in _runs14])

df_by_run_2h  = {}
sums_by_run_2h = {}   # run -> (SA, NV) at k2h, for the §14b bootstrap
for run, path in _runs14:
    pred = torch.load(path, map_location="cpu", weights_only=False)
    if pred["preds"].shape[2] != N:
        print(f"  skipping {run}: {pred['preds'].shape[2]} stations != {N} - "
              f"different EXCLUDE/network, not comparable on this station grid.")
        continue
    try:
        k2h = lead_index(pred, 120)
    except ValueError as e:
        print(f"  skipping {run}: {e}")
        continue
    df_by_run_2h[run]   = per_station_mae_table(pred, coords, keep_mask=KEEP, delta_idx=k2h)
    sums_by_run_2h[run] = _window_sums(pred, keep_mask=KEEP, delta_idx=k2h)
    del pred

### 14a — Per-model MAE maps @ 2 h

In [ ]:
if df_by_run_2h:
    _row_labels14 = list(df_by_run_2h.keys())
    draw_dem_grid(
        row_labels=_row_labels14,
        col_specs=_col_specs12,   # same (var, col, unit, cmap, (vmin,vmax)) as §12
        get_df=lambda rlab, clab: df_by_run_2h[rlab],
        suptitle="MAE by station @ 2 h lead - all runs, mr0.00, shared colour scale",
        save_path=os.path.join(RESULTS_ROOT, "map_mae_2h_by_model.png"),
        size=50,
    )
else:
    print("No comparable runs found for the §14a grid.")

### 14b — Winner map: which model has the lowest MAE at each station @ 2 h

Bootstrap: for each station and variable, resample that model's windows
`N_BOOT` times (independently per model, since different runs are not guaranteed
to share the same window set/order) to get a distribution of the runner-up's MAE
minus the winner's MAE. The win is marked **significant** (black ring) if that
difference's confidence interval excludes zero — i.e. the runner-up is not
plausibly just as good. Unmarked stations are effectively a tie between the top
two models.

In [ ]:
if len(df_by_run_2h) >= 2:
    boot_by_run = {run: _bootstrap_ci_from_sums(sa, nv, n_boot=N_BOOT, seed=abs(hash(run)) % (2**31))
                  for run, (sa, nv) in sums_by_run_2h.items()}   # run -> (n_boot, N, 5) normalised
    _runs_list = list(df_by_run_2h.keys())
    _run_color = {run: c for run, c in zip(_runs_list, plt.cm.tab10.colors)}

    WINNER = {}   # var -> DataFrame(station_idx, e_lv95, n_lv95, winner, sig)
    for v, (var, col, unit, cmap, (lo, hi)) in enumerate(VARIABLES):
        win_run, win_sig = [], []
        for n in range(N):
            maes = {run: df_by_run_2h[run][col].values[n] for run in _runs_list}
            valid = {r: m for r, m in maes.items() if not np.isnan(m)}
            if not valid:
                win_run.append(None); win_sig.append(False); continue
            ranked = sorted(valid.items(), key=lambda kv: kv[1])
            best_run = ranked[0][0]
            if len(ranked) > 1:
                second_run = ranked[1][0]
                # runner-up minus winner, in normalised space (both bootstrapped independently)
                d = boot_by_run[second_run][:, n, v] - boot_by_run[best_run][:, n, v]
                lo_d = np.nanpercentile(d, (1 - CI) / 2 * 100)
                sig = lo_d > 0   # runner-up reliably worse than winner
            else:
                sig = True
            win_run.append(best_run); win_sig.append(sig)

        wdf = coords[['station_idx', 'e_lv95', 'n_lv95']].copy()
        wdf['winner'] = win_run
        wdf['sig']    = win_sig
        WINNER[var] = wdf

    fig, axes = plt.subplots(1, len(VARIABLES), figsize=(6 * len(VARIABLES), 6))
    if len(VARIABLES) == 1:
        axes = [axes]
    for ax, (var, col, unit, cmap, (lo, hi)) in zip(axes, VARIABLES):
        draw_dem(ax, dem_bg, dem_fg, switzerland)
        wdf = WINNER[var]
        for run in _runs_list:
            sub = wdf[wdf['winner'] == run].dropna(subset=['e_lv95', 'n_lv95'])
            if len(sub):
                ax.scatter(sub['e_lv95'], sub['n_lv95'], color=_run_color[run],
                          s=55, edgecolors='white', linewidths=0.4, zorder=8, label=run)
        mark_significant(ax, wdf, 'sig', size=85)
        ax.set_title(f'{var} - winner @ 2h', fontsize=11)

    handles = [mlines.Line2D([], [], marker='o', linestyle='', color=_run_color[r], label=r)
              for r in _runs_list]
    handles.append(mlines.Line2D([], [], marker='o', linestyle='', markerfacecolor='none',
                                 markeredgecolor='black', markersize=9, label=f'significant @ {CI:.0%}'))
    fig.legend(handles=handles, loc='lower center', ncol=min(len(handles), 6),
              fontsize=9, bbox_to_anchor=(0.5, -0.06))
    fig.suptitle('Winner-takes-station map @ 2 h lead, by variable  '
                '(black ring = win is significant, not a coin flip)', fontsize=13, y=1.03)
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_ROOT, "map_winner_by_variable_2h.png"), dpi=150, bbox_inches="tight")
    plt.show()

    print("Win counts by model (all 5 variables pooled):")
    for run in _runs_list:
        n_wins = sum((WINNER[var]['winner'] == run).sum() for var, *_ in VARIABLES)
        n_sig  = sum(((WINNER[var]['winner'] == run) & WINNER[var]['sig']).sum() for var, *_ in VARIABLES)
        print(f"  {run:22s} {n_wins:4d} station-variable wins  ({n_sig} significant)")
else:
    print("Need at least 2 comparable runs for the §14b winner map.")

---
## 15 — Variance / uncertainty maps at fixed lead times: v30-nll vs v27

Two different notions of "spread", both mapped at the same three fixed lead
times (30 min / 2 h / 6 h) used in §12/§14, so they line up directly against
the MAE maps above:

- **§15a — v30-nll predicted variance.** v30-nll is the one run trained with
  `use_nll_loss=True` (Gaussian NLL), so its `predictions.pt` carries an extra
  `log_var` tensor — the model's own per-prediction variance estimate, not
  derived from residuals. Mapped here as the mean predicted variance per
  station (physical units²), i.e. how uncertain the model *says* it is.
- **§15b — v27 error variance (std).** v27 has no variance head (plain Huber
  loss), so the analogous quantity is the empirical spread of its own errors:
  `std(pred - target)` per station, physical units, i.e. how uncertain it
  *turns out* to be. Plotted as std (not variance) so it's directly comparable
  in scale to the MAE maps elsewhere in this notebook.

These are not the same axis — §15a is the model's calibrated uncertainty
(only meaningful if v30-nll's NLL is well-calibrated), §15b is realised error
dispersion (always meaningful, any model) — so they are shown as two separate
grids rather than overlaid.

In [ ]:
def per_station_variance_table(pred_dict, coords, keep_mask=None, delta_idx=None, source='error'):
    """
    Per-station variance (physical units^2) at a single fixed lead-time slot.

    source='error': empirical variance of the residual (pred - target) across
        valid test windows, per station/variable - the spread of the model's
        own errors. Works for any run, doesn't need a predicted uncertainty.
    source='model': mean of the model's predicted variance (exp(log_var))
        across valid windows - the model's own uncertainty estimate, only
        available for runs trained with use_nll_loss=True (carries a
        `log_var` tensor in predictions.pt, same shape as `preds`).

    Returns a DataFrame merged with coords, with `{var}_var` (physical
    variance) and `{var}_std` (physical std, sqrt of the variance) per
    variable, plus `{var}_n_samples`.
    """
    if delta_idx is None:
        raise ValueError("per_station_variance_table needs a fixed delta_idx (lead-time slot) - "
                         "variance at a single lead time, not averaged across lead times.")

    Mk = pred_dict['masks'][:, delta_idx, :, :5] > 0.5
    mask = Mk.numpy()
    if keep_mask is not None:
        mask = mask & keep_mask[None, :, :]
    n_valid = mask.sum(axis=0)                                    # (N, 5)

    if source == 'model':
        if 'log_var' not in pred_dict:
            raise ValueError("source='model' requires a 'log_var' tensor in pred_dict "
                             "(only runs trained with use_nll_loss=True have one).")
        var_norm = pred_dict['log_var'][:, delta_idx, :, :].exp().numpy()     # (M, N, 5)
        sum_var  = (var_norm * mask).sum(axis=0)
        var_norm_mean = np.divide(sum_var, n_valid,
                                  out=np.full(sum_var.shape, np.nan), where=n_valid > 0)
    elif source == 'error':
        P = pred_dict['preds'][:, delta_idx, :, :].numpy()
        T = pred_dict['targets'][:, delta_idx, :, :5].numpy()
        err = P - T                                                # (M, N, 5) normalised
        mean_err = np.divide((err * mask).sum(axis=0), n_valid,
                             out=np.full((mask.shape[1], mask.shape[2]), np.nan), where=n_valid > 0)
        sq_dev = (err - mean_err[None, :, :]) ** 2
        sum_sq_dev = (sq_dev * mask).sum(axis=0)
        var_norm_mean = np.divide(sum_sq_dev, n_valid,
                                  out=np.full(sum_sq_dev.shape, np.nan), where=n_valid > 0)
    else:
        raise ValueError(f"unknown source {source!r} - use 'error' or 'model'")

    var_phys = var_norm_mean * (STD ** 2)                          # (N, 5) physical units^2

    df = coords[['station_idx', 'nat_abbr', 'e_lv95', 'n_lv95', 'station_height']].copy()
    for v, (var, col, unit, *_r) in enumerate(VARIABLES):
        df[f'{var}_var']       = var_phys[:, v]
        df[f'{var}_std']       = np.sqrt(np.clip(var_phys[:, v], 0, None))
        df[f'{var}_n_samples'] = n_valid[:, v]
    return df


def _shared_bounds(tables, col, lo_q=0.02, hi_q=0.98):
    """(vmin, vmax) pooled across a list of per-lead-time DataFrames, so all
    three lead-time panels for a variable share one colour scale."""
    vals = pd.concat([t[col] for t in tables]).dropna()
    if len(vals) == 0:
        return 0.0, 1.0
    lo, hi = float(vals.quantile(lo_q)), float(vals.quantile(hi_q))
    return (0.0, hi) if lo >= 0 else (lo, hi)   # variance/std are >= 0 - floor at 0


LEAD_LABELS_15 = LEAD_LABELS   # reuse the (label, minutes) list from S12: 30 min / 2 h / 6 h

### 15a — v30-nll: mean predicted variance, physical units², by lead time

In [ ]:
_run30 = None
for _run, _path in _runs14:
    if _run == "v30-nll":
        _run30 = _path
        break

if _run30 is None:
    print("Run 'v30-nll' not found under test_results/*/best_mr0.00/predictions.pt - skipping S15a.")
else:
    pred_v30 = torch.load(_run30, map_location="cpu", weights_only=False)
    if "log_var" not in pred_v30:
        print("v30-nll predictions.pt has no 'log_var' tensor - this dump was not produced "
              "with use_nll_loss=True. Skipping S15a.")
    else:
        _idx_v30 = {lab: lead_index(pred_v30, mins) for lab, mins in LEAD_LABELS_15}
        var_by_lead_v30 = {
            lab: per_station_variance_table(pred_v30, coords, keep_mask=KEEP, delta_idx=k, source='model')
            for lab, k in _idx_v30.items()
        }
        _tables15a = list(var_by_lead_v30.values())
        _col_specs15a = [
            (var, f'{var}_var', f'{unit}\u00b2', 'inferno_r',
             _shared_bounds(_tables15a, f'{var}_var'))
            for var, _, unit, _, _ in VARIABLES
        ]

        draw_dem_grid(
            row_labels=[lab for lab, _ in LEAD_LABELS_15],
            col_specs=_col_specs15a,
            get_df=lambda rlab, clab: var_by_lead_v30[rlab],
            suptitle="v30-nll - predicted variance by station at fixed lead times "
                     "(Gaussian-NLL uncertainty head, shared colour scale per variable)",
            save_path=os.path.join(RESULTS_ROOT, "map_v30nll_predicted_variance_by_leadtime.png"),
            size=50,
        )
        del pred_v30

### 15b — v27: empirical error std, physical units, by lead time

In [ ]:
_run27 = None
for _run, _path in _runs14:
    if _run == "v27":
        _run27 = _path
        break

if _run27 is None:
    print("Run 'v27' not found under test_results/*/best_mr0.00/predictions.pt - skipping S15b.")
else:
    pred_v27 = torch.load(_run27, map_location="cpu", weights_only=False)
    _idx_v27b = {lab: lead_index(pred_v27, mins) for lab, mins in LEAD_LABELS_15}
    std_by_lead_v27 = {
        lab: per_station_variance_table(pred_v27, coords, keep_mask=KEEP, delta_idx=k, source='error')
        for lab, k in _idx_v27b.items()
    }
    _tables15b = list(std_by_lead_v27.values())
    _col_specs15b = [
        (var, f'{var}_std', unit, 'inferno_r',
         _shared_bounds(_tables15b, f'{var}_std'))
        for var, _, unit, _, _ in VARIABLES
    ]

    draw_dem_grid(
        row_labels=[lab for lab, _ in LEAD_LABELS_15],
        col_specs=_col_specs15b,
        get_df=lambda rlab, clab: std_by_lead_v27[rlab],
        suptitle="v27 - empirical error std by station at fixed lead times "
                 "(std of pred - target across test windows, shared colour scale per variable)",
        save_path=os.path.join(RESULTS_ROOT, "map_v27_error_std_by_leadtime.png"),
        size=50,
    )

    print("Median error std by lead time (v27):")
    for lab, _ in LEAD_LABELS_15:
        t = std_by_lead_v27[lab]
        vals = ", ".join(f"{var}={t[f'{var}_std'].median():.3f}" for var, *_ in VARIABLES)
        print(f"  {lab}: {vals}")

    del pred_v27